# [수상작 리뷰] 문맥 기반 문장 순서 예측 1위 솔루션 분석

### **코드 흐름 요약:**
이 수상작은 LLM의 복잡한 추론 능력을 묻기보다는, Decoder-only 모델의 본질인 인과적 언어 모델링(Causal Language Modeling) 능력을 활용한 직관적이고 강력한 접근법을 보여줌.

1. 데이터 전처리 및 토큰화
* 주어진 4개의 문장을 정답 순서대로 배치한 뒤, 문장 사이에 → 스페셜 토큰을 삽입하여 하나의 시퀀스(correct_order)로 만듦.
* tokenizer를 이용해 모델별 최대 시퀀스 길이를 동적으로 계산.

2. 추론: Perplexity (PPL) 기반 순위 매기기
* 4개의 문장으로 만들 수 있는 모든 경우의 수(순열 24가지)를 생성.
* 별도의 분류(Classification) 헤드를 달지 않고, 24개의 문장 조합을 모델에 입력해 각 시퀀스의 Perplexity(PPL, 텍스트의 자연스러운 정도) 계산.
* PPL이 가장 낮은(가장 자연스러운) 문장 순서를 정답으로 예측.

3. 모델 학습 (Fine-tuning & QDoRA)
* 중소형 모델(< 14B): Causal LM 목적으로 일반적인 Full Fine-tuning을 진행.
* 대형 모델(≥ 14B): VRAM 절약을 위해 4-bit 양자화와 QDoRA (Quantized DoRA)를 적용. 이를 위해 8-bit PagedAdamW 옵티마이저를 직접 커스텀하여 사용.

* 앙상블 (FLAML-Rank)
* 단일 모델의 PPL 예측값을 그대로 쓰지 않고, 여러 모델(EXAONE, Gemma, Qwen 등)이 예측한 24개 순열의 PPL 값들을 z-score로 정규화.
* 정규화된 PPL 피처들을 FLAML의 AutoML-Rank(LightGBM 기반) 모델에 입력하여, 최종적으로 가장 정답일 확률이 높은 순열을 랭킹(Learning to Rank) 방식으로 뽑아냄.

### **새롭게 알게 된 내용 및 배울 점**
단순히 문제 풀이를 위한 라이브러리 호출을 넘어, PyTorch 모델의 내부 로직을 직접 뜯어보고 목적에 맞게 모델과 옵티마이저를 커스텀한 구현력을 새롭게 알게 되었다.

특히, HuggingFace Trainer가 계산해주는 Loss에 의존하지 않고, 모델의 logits를 뽑아내어 직접 한 스텝씩 Shift하며 토큰 단위의 Cross Entropy Loss를 구하는 방식을 구현한 것이 배울 점이라고 느꼈다.

### **주요 코드 3줄 이상**

In [ ]:
# Logit Shift를 통한 문장별 PPL 직접 계산.
# 다음 토큰 예측용으로 shift (마지막 토큰 로짓과 첫 번째 토큰 라벨 제외)
shift_logits = logits[:, :-1, :].contiguous()    # (B, L-1, V)
shift_labels = input_ids[:, 1:].contiguous()     # (B, L-1)
shift_mask   = attention_mask[:, 1:].contiguous()# (B, L-1)

In [ ]:
# 토큰별 NLL(loss) 계산 (reduction='none'으로 토큰별 손실 유지)
vocab_size = shift_logits.size(-1)
loss_per_token = F.cross_entropy(
    shift_logits.view(-1, vocab_size),
    shift_labels.view(-1),
    reduction="none"
).view_as(shift_labels)

In [ ]:
# 파라미터 그룹 분리 및 커스텀 옵티마이저 (DoRA)
# 모델의 모든 파라미터를 순회하면서 그룹 분류
for name, param in model.named_parameters():
    if param.requires_grad:
        if 'lora_magnitude_vector' in name:
            magnitude_params.append(param)  # DoRA 크기 성분
        elif 'lora_A' in name or 'lora_B' in name:
            direction_params.append(param)  # DoRA 방향 성분
        else:
            other_params.append(param)      # 기타

In [ ]:
# 그룹별로 다른 학습률 적용 (Magnitude는 Direction의 1/5 수준으로 세팅)
optimizer_grouped_parameters = [
    {"params": direction_params, "lr": base_lr, "weight_decay": weight_decay_value},
    {"params": magnitude_params, "lr": base_lr * magnitude_lr_ratio, "weight_decay": 0},
    {"params": other_params, "lr": base_lr, "weight_decay": weight_decay_value}
]